# Level 1 · How are the School's rooms used today, and does the new model fit?

**UAB THE HACK! 2026 · Escola d'Enginyeria challenge.** Team: sudo-bros.

The new model: **1st and 2nd year in the morning (8:30-14:30); 3rd, 4th, double degrees and master's in the
afternoon (14:30-21:00)**, always keeping **2 ordinary classrooms free** in every hour (so 19 of the 21 are usable).

How to read this notebook: every number comes from a function in [`analisi.py`](analisi.py), which uses the same
conventions as the official verifier (semester by date, shift by start time, a shared class counts once, double-degree
groups need no room, PLAB labs are fixed). Run it top to bottom (*Run All*). The data must be in
`repte-horaris-escola/paquet/` (see `SETUP.md`).

The five findings are summarised in [`TROBALLES.md`](TROBALLES.md).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from analisi import *

pd.set_option('display.width', 160)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
s, a = carrega()
s2 = s[s['smt'] == 2]
print(f"{len(s):,} sessions ({(s['smt'] == 1).sum():,} in semester 1, {len(s2):,} in semester 2) · {len(a)} rooms")
a.groupby('tipus')['capacitat'].agg(['count', 'min', 'max', 'sum'])

## 1. Room occupancy by day and hour

For each room and each hour slot of the week: **in what share of the teaching weeks is it busy?**
(1.0 = every week, 0 = never). Only sessions that have a room in the data can be drawn, which matters a lot (section 2).

In [ ]:
def heatmap(o, title, ax):
    im = ax.imshow(o.values, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
    ax.set_yticks(range(len(o)), [f"{r} ({int(c) if c == c else '?'})" for r, c in
                                  zip(o.index, a.set_index('aula')['capacitat'].reindex(o.index))], fontsize=6)
    xt = [i for i, (d, f) in enumerate(o.columns) if f in (FRANGES[0], MATI_FI)]
    ax.set_xticks(xt, [f"{NOM_DIA[o.columns[i][0]]} {fmt(o.columns[i][1])}" for i in xt], fontsize=7, rotation=90)
    for i, (d, f) in enumerate(o.columns):
        if f == FRANGES[0] and i:
            ax.axvline(i - .5, color='k', lw=1)
    ax.set_title(title)
    return im

fig, axs = plt.subplots(1, 2, figsize=(15, 9), sharey=False)
for smt, ax in zip((1, 2), axs):
    im = heatmap(ocupacio_aules(s, a, smt), f'Semester {smt}: share of weeks each room is busy', ax)
fig.colorbar(im, ax=axs, shrink=.5, label='share of weeks busy')
plt.show()

In [ ]:
r1, r2 = resum_aules(s, a, 1), resum_aules(s, a, 2)
ord1, ord2 = r1[r1['tipus'] == 'Aules docents'], r2[r2['tipus'] == 'Aules docents']
print('Average occupancy of the 21 ordinary classrooms (from the rooms in the data):')
print(f"  semester 1: morning {ord1['ocup_mati'].mean():.0%}, afternoon {ord1['ocup_tarda'].mean():.0%}")
print(f"  semester 2: morning {ord2['ocup_mati'].mean():.0%}, afternoon {ord2['ocup_tarda'].mean():.0%}")
print('\nBusiest and emptiest rooms in semester 2:')
pd.concat([r2.head(5), r2[r2['tipus'] == 'Aules docents'].tail(5)]).round(2)

**Reading it:** in semester 1 the rooms look reasonably used (ordinary classrooms ~50 % in the morning). In
semester 2 they look almost empty (~22 % morning, ~4 % afternoon), and some 90-seat rooms (Q2/1009, Q2/1013) appear
nearly unused. **That is not real:** in semester 2 most sessions have no room in the data, so they are invisible here.
The next section measures the real demand.

## 2. Sessions without a room (semester 2), and how many rooms they need

5,722 of the 7,739 sessions of semester 2 (74 %) have an empty `aula_partida`. Not all of them need a room:

In [ ]:
m = sense_aula(s, 2)
print(m.to_string(), f"\nTotal without room: {m.sum():,} of {len(s2):,} ({m.sum() / len(s2):.0%})")
ax = m.plot.barh(color=['#c0392b', '#7f8c8d', '#95a5a6'], figsize=(7, 2.2))
ax.set_xlabel('sessions'); ax.set_title('Semester 2 sessions without a room, by reason'); ax.invert_yaxis()
plt.show()

To know how many rooms are really needed we count, for every date and hour slot, **how many classes need a
classroom at the same time** with today's timetable (with or without a room in the data; one per physical class;
PLAB, double degrees and classes that use a lab or computer room left out).

In [ ]:
g2 = simultanies(s, 2)
pic = g2.groupby('franja')[['total', 'amb_aula', 'sense_aula']].max()
pic.index = [fmt(f) for f in pic.index]
ax = pic['total'].plot.bar(color='#e67e22', figsize=(9, 3.5), label='classes needing a classroom (peak day)')
ax.axhline(19, color='k', ls='--', label='19 usable ordinary rooms (21 - 2 reserve)')
ax.axhline(19 + 8, color='grey', ls=':', label='+ 8 seminars (small groups only)')
ax.set_ylabel('simultaneous classes'); ax.set_xlabel('hour slot'); ax.legend(fontsize=8)
ax.set_title('Semester 2, today: peak simultaneous classroom demand per hour slot')
plt.show()
over = g2[g2['total'] > 19]
print(f"Slots (date x hour) above 19: {len(over)} of {len(g2)}; maximum {g2['total'].max()} classes at once")
over.sort_values('total', ascending=False).head(6)

**Reading it:** the real demand is concentrated at **10:30-13:30 and 16:30**. Even today, some slots need more
than the 19 usable ordinary rooms (up to 21): small groups have to go to seminars. The rooms that look empty in
section 1 are in fact needed: the data simply doesn't say which session goes where.

## 3. Who is in the wrong shift today?

A session is in the wrong shift when its start time (before/after 14:30) doesn't match `torn_requerit`. A session
shared by two cohorts counts for both in the per-degree table. **Movable** = not fixed (fixed = GEI and GEET+GEST
1st/2nd year, and PLAB labs: they can't move, so the verifier marks them as inherited).

In [ ]:
ft = s2['fora_torn']
mov = ft & (s2['fixada'] == '')
print(f"Semester 2: {ft.sum():,} sessions in the wrong shift; {mov.sum():,} movable, {(ft & (s2['fixada'] != '')).sum():,} fixed")
print('Movable ones, by where they must go:', s2.loc[mov, 'torn_requerit'].map({'tarda': 'to the afternoon', 'mati': 'to the morning'}).value_counts().to_dict())
print(f"Of these, {(mov & s2['doble']).sum()} are double-degree sessions taught today with a morning group (exception candidates)")
t = fora_de_torn(s, 2)
t = t[t['fora_torn'] > 0]
t

In [ ]:
t2 = t.assign(cohort=t['titulacio'] + ' ' + t['curs_c']).set_index('cohort')
t2['fixed'] = t2['fora_torn'] - t2['fora_torn_movibles']
ax = t2[['fora_torn_movibles', 'fixed']].rename(columns={'fora_torn_movibles': 'movable', 'fixed': 'fixed (inherited)'}) \
    .plot.barh(stacked=True, color=['#c0392b', '#bdc3c7'], figsize=(8, 5))
for i, (n, p) in enumerate(zip(t2['fora_torn'], t2['pct_fora_torn'])):
    ax.text(n + 10, i, f'{p:.0f} %', va='center', fontsize=7)
ax.invert_yaxis(); ax.set_xlabel('sessions in the wrong shift (semester 2)')
ax.set_title('Sessions in the wrong shift by degree and year (% of that cohort\'s sessions)')
plt.show()

**Reading it:** the change is mostly **3rd and 4th year moving to the afternoon**: GEI 3r/4t (~90 % of their
sessions), GEST 3r (99 %), GEET 3r (98 %), GEQ 3r (100 %). In the other direction, GCIS 2n and GIA 1r/2n are today
entirely in the afternoon and must come to the morning.

The **fixed** ones are a structural limit of the model: GEI 1r/2n groups that today are in the afternoon "are not
touched", so they stay in the wrong shift (600 class sessions); and fixed PLAB labs stay where they are (GEI 3r/4t labs in the morning,
GIA 1r/2n labs in the afternoon). The verifier doesn't count them (inherited), but they are real exceptions to rule 1:

In [ ]:
f = s2[s2['fora_torn'] & (s2['fixada'] != '')]
f.groupby(['titulacions_curs', 'tipus', 'torn_requerit']).size().rename('fixed sessions in the wrong shift') \
    .reset_index().sort_values('fixed sessions in the wrong shift', ascending=False)

## 4. Does the morning fit?

If all 1st and 2nd year go to the morning, compare **room-hours demanded** with **room-hours offered**:
19 rooms × 6 hours × teaching days of the week. Two definitions of demand:

- **classroom only** (ours): classes that need an ordinary classroom or seminar. Classes that today use a computer
  room or integrated lab can keep using it.
- **statement's definition**: all 1st/2nd-year sessions except PLAB and double degrees (gives the 67-70 % quoted in
  the statement).

In [ ]:
h = hores_aula_setmana(s, 2, 'mati')
full = h[h['normal']]   # 5 teaching days and nothing in calendari.csv (no holiday, stoppage or assessment)
h_st = (demanda_aula(s2, inclou_practiques=True).query("torn_requerit == 'mati'").groupby('setmana')['durada_min'].sum() / 60)
full = full.assign(statement_pct=(100 * h_st.reindex(full.index) / full['oferta_h']).round(1))
print(f"Normal weeks ({len(full)}): morning use {full['ocupacio_pct'].min()}-{full['ocupacio_pct'].max()} % (classroom only), "
      f"{full['statement_pct'].min()}-{full['statement_pct'].max()} % (statement's definition); "
      f"today, with only those already in the morning: {full['ocupacio_ja_al_torn_pct'].min()}-{full['ocupacio_ja_al_torn_pct'].max()} %")
ax = h[['demanda_ja_al_torn_h', 'demanda_h']].rename(columns={'demanda_ja_al_torn_h': 'already in the morning today',
                                                              'demanda_h': 'all 1st/2nd year (new model)'}) \
    .plot.bar(figsize=(10, 3.5), color=['#f5b041', '#c0392b'])
ax.plot(range(len(h)), h['oferta_h'], 'k_', ms=18, mew=2, label='offer: 19 rooms × 6 h × days')
ax.set_ylabel('room-hours per week'); ax.legend(fontsize=8)
ax.set_title('Semester 2, morning: room-hours demanded vs offered, per week')
plt.show()
h_t = hores_aula_setmana(s, 2, 'tarda')
print(f"Afternoon (3r, 4t, master's; 19 rooms × 6.5 h): {h_t.loc[h_t['normal'], 'ocupacio_pct'].min()}-"
      f"{h_t.loc[h_t['normal'], 'ocupacio_pct'].max()} % in normal weeks")

Room-hours are an average. What really limits is the **peak**: how many classes at the same time. With today's
hours, the 1st/2nd-year classes already in the morning reach **15 at once** (statement: "15 of 19"). The ones that
come from the afternoon have no hour yet (that is task 5.2), but the weekly total tells us the average they leave:

In [ ]:
w = '2027-W06'   # a normal full week
d = demanda_aula(s2[s2['setmana'] == w]).query("torn_requerit == 'mati'")
per_dia = (d.groupby('dia')['durada_min'].sum() / 60).reindex(DIES)
print(f"Week {w}: {d['durada_min'].sum() / 60:.0f} room-hours of morning classes = on average "
      f"{d['durada_min'].sum() / 60 / 30:.1f} classes at once over the 30 morning hours (limit 19)")
print('Per weekday (keeping today\'s days), average classes at once:', (per_dia / 6).round(1).rename(NOM_DIA).to_dict())
gm = simultanies(s, 2, lambda x: (x['torn_requerit'] == 'mati') & (x['torn_actual'] == 'mati'))
p = gm.groupby(['dia', 'franja'])['total'].max().unstack().reindex(DIES)
fig, ax = plt.subplots(figsize=(6, 2.6))
im = ax.imshow(p.values, cmap='YlOrRd', vmin=0, vmax=19)
for i in range(p.shape[0]):
    for j in range(p.shape[1]):
        ax.text(j, i, int(p.values[i, j]), ha='center', va='center', fontsize=8)
ax.set_xticks(range(p.shape[1]), [fmt(f) for f in p.columns]); ax.set_yticks(range(5), [NOM_DIA[x] for x in DIES])
ax.set_title('1st/2nd year already in the morning: peak classes at once (of 19)', fontsize=9)
plt.show()

**Does size matter?** A counting check for the same week: for each room size N, the morning classes that need at
least N seats against the rooms that have at least N seats (the 2 reserve rooms taken from the smallest). If demand
went over offer for some N, the morning could not fit whatever the timetable.

In [ ]:
mm = mida_mati(s, a, 2, w)
display(mm)
ax = mm.plot.bar(x='seats_at_least', y=['demand_h', 'offer_h'], color=['#c0392b', '#d5dbdb'], figsize=(8, 3))
ax.set_xlabel('rooms with at least N seats'); ax.set_ylabel('room-hours, week ' + w)
ax.set_title('Morning: demand vs offer by room size')
plt.show()

**Reading it:** **yes, the morning fits, but the margin is the peak, not the total.** The new model uses
59-64 % of the morning room-hours in normal weeks (67-70 % with the statement's definition) and no size class goes above 33 % of
the rooms big enough for it. Each student subgroup has at most 20 h of class a week against the 30 morning hours.
The risk is Tuesday and the 10:30-13:30 slots, already at 13-15 of 19 today: the classes that move to the morning
have to be spread over the quieter days and hours (Wednesday, Friday, 8:30). The afternoon, on the other hand, is
left with lots of spare capacity (~35-40 %).

## 5. Groups with more students than seats in today's room

One physical class (students of a shared session summed) against the room's capacity, semester 2.

In [ ]:
sa = sobre_aforament(s, a, 2)
print(f"{sa['sessions'].sum()} sessions in {len(sa)} groups over capacity; "
      f"{sa.loc[sa['fixada'] == '', 'sessions'].sum()} of them movable (the rest are fixed GEI groups: inherited)")
big = a[a['ordinaria'] == 1].sort_values('capacitat', ascending=False)[['aula', 'capacitat']].head(6)
print('Biggest ordinary rooms:', ', '.join(f"{r.aula} ({r.capacitat})" for r in big.itertuples()))
sa

**Reading it:** only two movable groups are over capacity: **Física TE/21 (GEQ 1r), 125 students in Q4/0011 (117)**
and **Sistemes i Tecnologies Web TE/450 (GEI 3r/4t), 103 in Q3/0003 (100)**. Both have bigger rooms available
(Q1/1011 160, Q3/1007 and Q3/1011 130). The other four are fixed GEI PAUL groups 1-2 students over a 48-50 seat room.

## 6. Plan vs reality

The statement suggests comparing with `validacio/` (real occupancy of one October week). That CSV is **not in the
package we received** (only its README), so this comparison is left out.